In [ ]:
from google.colab import files
import pandas as pd
import re
uploaded = files.upload()
df = pd.read_csv("resumeJD2_pairs.csv")
print(df.shape)
df.head()
df.info()
df.isnull().sum()
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates()
df["resume_text"] = df["resume_text"].str.strip()
df["job_description"] = df["job_description"].str.strip()

def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    return text.strip()
df["resume_text"] = df["resume_text"].apply(clean_text)
df["job_description"] = df["job_description"].apply(clean_text)

print(df["match_label"].value_counts())
print(df["match_label"].unique())
print(df["match_score"].describe())

In [ ]:
import re
from google.colab import files
import pandas as pd
uploaded = files.upload()

# Read dataset
df = pd.read_csv("resumeJD2_pairs.csv")

# -------------------------
# Text Cleaning Function
# -------------------------

def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Convert to lowercase
    text = text.lower()

    # Remove emails
    text = re.sub(r'\S+@\S+', ' ', text)

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', ' ', text)

    # Remove LinkedIn URLs
    text = re.sub(r'linkedin\.com/\S+', ' ', text)

    # Remove phone numbers
    text = re.sub(r'\+?\d[\d\s\-\(\)]{8,}\d', ' ', text)

    # Remove special characters
    text = re.sub(r'[^a-z0-9\s]', ' ', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text)

    return text.strip()


# -------------------------
# Metadata Removal
# -------------------------

REMOVE_PATTERNS = [
    r'name\s*:',
    r'email\s*:',
    r'phone\s*:',
    r'mobile\s*:',
    r'linkedin\s*:',
    r'github\s*:',
    r'address\s*:'
]

def remove_metadata(text):
    for pattern in REMOVE_PATTERNS:
        text = re.sub(pattern, ' ', text)
    return text


# -------------------------
# Apply Cleaning
# -------------------------

df["resume_text"] = df["resume_text"].apply(clean_text)
df["job_description"] = df["job_description"].apply(clean_text)

df["resume_text"] = df["resume_text"].apply(remove_metadata)
df["job_description"] = df["job_description"].apply(remove_metadata)

# -------------------------
# Remove Very Short Records
# -------------------------

df = df[
    (df["resume_text"].str.len() > 100) &
    (df["job_description"].str.len() > 100)
]

# -------------------------
# Check Results
# -------------------------

print("Final Shape:", df.shape)

print("\nMatch Label Distribution:")
print(df["match_label"].value_counts())

print("\nMatch Score Statistics:")
print(df["match_score"].describe())
df.to_csv("cleaned_resumeJD2_pairs.csv", index=False)

files.download("cleaned_resumeJD2_pairs.csv")

print("\nDataset saved successfully!")

